# InsPLAD LVLM Benchmark — Multi-model Colab baselines

MMAD처럼 **동일한 QA / 동일한 prompt / 동일한 decoding 조건**에서 여러 general-purpose LVLM을 비교하는 Colab runner입니다.

기본 공개 baseline 모델:
- `Qwen/Qwen3-VL-8B-Instruct`
- `OpenGVLab/InternVL3_5-8B`
- `openbmb/MiniCPM-V-4_5`
- `llava-hf/llava-onevision-qwen2-7b-ov-hf`

각 모델은 한 번에 하나씩 GPU에 올린 뒤 **Zero-shot 20 → One-shot 20 → Structured 20**을 실행하고 즉시 unload합니다. 마지막에 모델별 metric과 leaderboard CSV를 생성합니다.

> 먼저 20-sample smoke test만 수행합니다. 전체 6,481 / 6,417 inference는 이 비교가 정상 동작한 뒤 진행합니다.


In [ ]:
# 1) Colab 환경 + 저장소 준비
import os, sys, subprocess, json, gc, shutil, time, traceback
from pathlib import Path

if not Path('/content').exists():
    raise RuntimeError('Google Colab 실행을 기준으로 작성된 notebook입니다.')

REPO_URL = 'https://github.com/arnold6444/lvlm_insplad.git'
BRANCH = 'chatgpt/insplad-benchmark-v1'
REPO_DIR = Path('/content/lvlm_insplad')

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'transformers>=5.0.0', 'accelerate>=1.0', 'pillow', 'pandas',
                'matplotlib', 'tqdm', 'sentencepiece'], check=True)

if not REPO_DIR.exists():
    subprocess.run(['git','clone','--branch',BRANCH,'--single-branch',REPO_URL,str(REPO_DIR)], check=True)
else:
    subprocess.run(['git','-C',str(REPO_DIR),'fetch','origin',BRANCH], check=True)
    subprocess.run(['git','-C',str(REPO_DIR),'switch',BRANCH], check=True)
    subprocess.run(['git','-C',str(REPO_DIR),'pull','--ff-only','origin',BRANCH], check=True)

subprocess.run([sys.executable,'-m','pip','install','-q','-e',f'{REPO_DIR}[dev]'], check=True)
os.chdir(REPO_DIR)
subprocess.run([sys.executable,'-m','pytest','-q'], check=True)
print('repo ready:', REPO_DIR)


In [ ]:
# 2) GPU 확인
import torch
print('torch:', torch.__version__)
print('cuda available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('Colab Runtime에서 GPU를 선택하세요.')
props = torch.cuda.get_device_properties(0)
VRAM_GB = props.total_memory / 1024**3
print('GPU:', props.name)
print(f'VRAM: {VRAM_GB:.1f} GB')
if VRAM_GB < 20:
    print('WARNING: 7B~8B 모델 FP16/BF16은 VRAM이 빡빡할 수 있습니다. 24GB+ GPU를 권장합니다.')


In [ ]:
# 3) Google Drive 연결 + InsPLAD ZIP 찾기
from google.colab import drive
drive.mount('/content/drive')
DRIVE_ROOT = Path('/content/drive/MyDrive')
UNSUP_NAME = 'unsupervised_anomaly_detection.zip'
SUP_NAME = 'supervised_fault_classification.zip'
COPY_ZIPS_TO_LOCAL = True

def find_drive_file(filename):
    direct = DRIVE_ROOT / filename
    if direct.exists(): return direct
    matches = list(DRIVE_ROOT.rglob(filename))
    if not matches: raise FileNotFoundError(filename)
    if len(matches) > 1: print('same-name files found; using:', matches[0])
    return matches[0]

unsup_drive = find_drive_file(UNSUP_NAME)
sup_drive = find_drive_file(SUP_NAME)
if COPY_ZIPS_TO_LOCAL:
    local = Path('/content/insplad_zips'); local.mkdir(exist_ok=True)
    unsup_zip, sup_zip = local/UNSUP_NAME, local/SUP_NAME
    for src,dst in [(unsup_drive,unsup_zip),(sup_drive,sup_zip)]:
        if not dst.exists() or dst.stat().st_size != src.stat().st_size:
            shutil.copy2(src,dst)
else:
    unsup_zip, sup_zip = unsup_drive, sup_drive
print(unsup_zip)
print(sup_zip)


In [ ]:
# 4) 동일한 20-sample QA 로드

def read_jsonl(path):
    return [json.loads(x) for x in Path(path).read_text(encoding='utf-8').splitlines() if x.strip()]

QA_SETS = {
    'zero_shot': read_jsonl('samples/anomaly_zero_shot_20.jsonl'),
    'one_shot': read_jsonl('samples/anomaly_one_shot_20.jsonl'),
    'structured': read_jsonl('samples/defect_20.jsonl'),
}
print({k: len(v) for k,v in QA_SETS.items()})


In [ ]:
# 5) smoke test에 필요한 이미지만 ZIP에서 /content로 추출
from zipfile import ZipFile

IMAGE_ROOT = Path('/content/insplad_smoke_images')
IMAGE_ROOT.mkdir(parents=True, exist_ok=True)

def safe_extract_member(zip_path, member, dataset_key):
    out = IMAGE_ROOT / dataset_key / member
    if out.exists(): return out
    out.parent.mkdir(parents=True, exist_ok=True)
    with ZipFile(zip_path) as zf:
        try:
            data = zf.read(member)
        except KeyError:
            normalized = member.replace('\\','/').lstrip('/')
            matches = [n for n in zf.namelist() if n.replace('\\','/').lstrip('/') == normalized]
            if not matches: raise
            data = zf.read(matches[0])
    out.write_bytes(data)
    return out

PATH_CACHE = {}
for task, rows in QA_SETS.items():
    for row in rows:
        if task in {'zero_shot','one_shot'}:
            PATH_CACHE[row['image']] = safe_extract_member(unsup_zip,row['image'],'unsup')
            if row.get('reference_image'):
                PATH_CACHE[row['reference_image']] = safe_extract_member(unsup_zip,row['reference_image'],'unsup')
        else:
            PATH_CACHE[row['image']] = safe_extract_member(sup_zip,row['image'],'sup')
print('extracted unique images:', len(PATH_CACHE))


## Baseline model registry

MMAD처럼 모델 계열을 나눠 비교하되, 이 notebook은 **Hugging Face에서 직접 실행 가능한 공개 모델**을 우선합니다. `MODELS_TO_RUN`만 바꾸면 일부 모델만 먼저 돌릴 수 있습니다.


In [ ]:
# 6) 비교 모델 registry
MODEL_REGISTRY = {
    'qwen3_vl_8b': {
        'model_id': 'Qwen/Qwen3-VL-8B-Instruct',
        'family': 'Qwen',
        'trust_remote_code': False,
    },
    'internvl3_5_8b': {
        'model_id': 'OpenGVLab/InternVL3_5-8B',
        'family': 'InternVL',
        'trust_remote_code': True,
    },
    'minicpm_v_4_5': {
        'model_id': 'openbmb/MiniCPM-V-4_5',
        'family': 'MiniCPM',
        'trust_remote_code': True,
    },
    'llava_onevision_7b': {
        'model_id': 'llava-hf/llava-onevision-qwen2-7b-ov-hf',
        'family': 'LLaVA',
        'trust_remote_code': False,
    },
}
MODELS_TO_RUN = list(MODEL_REGISTRY)
print(MODELS_TO_RUN)


In [ ]:
# 7) 모든 모델에 공통으로 적용할 prompt / output 규칙
ANOMALY_PROMPT = (
    'Inspect the target power-line asset image. Classify it as normal or anomalous. '
    'Return only one JSON object exactly like {"anomaly":"normal"} or {"anomaly":"anomalous"}. '
    'Do not add explanation.'
)
ONE_SHOT_PROMPT = (
    'Image 1 is a NORMAL reference of the same asset category. Image 2 is the TARGET image. '
    'Compare the target with the normal reference and classify only the TARGET as normal or anomalous. '
    'Return only one JSON object exactly like {"anomaly":"normal"} or {"anomaly":"anomalous"}. '
    'Do not add explanation.'
)

def structured_prompt(row):
    classes = ', '.join(row['valid_defect_classes'])
    return (
        'Inspect the target power-line asset. Predict both anomaly status and defect class. '
        f'Allowed defect_class values for this asset are: {classes}. '
        'Return only one JSON object with exactly two keys: '
        '{"anomaly":"normal|anomalous","defect_class":"<allowed class>"}. '
        'If normal, defect_class must be "normal". Do not add explanation.'
    )


In [ ]:
# 8) 공통 Hugging Face pipeline adapter
from transformers import pipeline
from tqdm.auto import tqdm

def make_chat(image_paths, prompt):
    content = [{'type':'image','url':str(p)} for p in image_paths]
    content.append({'type':'text','text':prompt})
    return [{'role':'user','content':content}]

def extract_generated_text(out):
    item = out[0] if isinstance(out, list) else out
    if isinstance(item, list): item = item[0]
    if isinstance(item, dict):
        value = item.get('generated_text', '')
    else:
        value = item
    if isinstance(value, str): return value.strip()
    if isinstance(value, list):
        texts=[]
        for v in value:
            if isinstance(v, dict):
                if isinstance(v.get('content'), str): texts.append(v['content'])
                elif isinstance(v.get('content'), list):
                    for c in v['content']:
                        if isinstance(c, dict) and c.get('text'): texts.append(str(c['text']))
        return '\n'.join(texts).strip()
    return str(value).strip()

def load_pipe(spec):
    return pipeline(
        'image-text-to-text',
        model=spec['model_id'],
        device_map='auto',
        dtype='auto',
        trust_remote_code=spec['trust_remote_code'],
    )

def infer_row(pipe, task, row):
    if task == 'zero_shot':
        images=[PATH_CACHE[row['image']]]; prompt=ANOMALY_PROMPT
    elif task == 'one_shot':
        images=[PATH_CACHE[row['reference_image']], PATH_CACHE[row['image']]]; prompt=ONE_SHOT_PROMPT
    else:
        images=[PATH_CACHE[row['image']]]; prompt=structured_prompt(row)
    chat = make_chat(images, prompt)
    out = pipe(text=chat, max_new_tokens=80, do_sample=False, return_full_text=False)
    return extract_generated_text(out)


In [ ]:
# 9) 모델별 60개 smoke inference + 즉시 저장 + unload
RUN_ROOT = DRIVE_ROOT / 'lvlm_insplad_results' / time.strftime('multimodel_smoke_%Y%m%d_%H%M%S')
RUN_ROOT.mkdir(parents=True, exist_ok=True)
RUN_STATUS=[]

def write_jsonl(path, rows):
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open('w', encoding='utf-8') as f:
        for r in rows: f.write(json.dumps(r, ensure_ascii=False)+'\n')

for model_key in MODELS_TO_RUN:
    spec=MODEL_REGISTRY[model_key]
    model_dir=RUN_ROOT/model_key; model_dir.mkdir(parents=True,exist_ok=True)
    print('\n===',model_key,spec['model_id'],'===')
    try:
        pipe=load_pipe(spec)
        for task, rows in QA_SETS.items():
            preds=[]
            for row in tqdm(rows, desc=f'{model_key}:{task}'):
                started=time.time()
                try:
                    output=infer_row(pipe,task,row)
                    err=None
                except Exception as exc:
                    output=''; err=f'{type(exc).__name__}: {exc}'
                preds.append({
                    'id': row['id'], 'output': output, 'error': err,
                    'latency_s': round(time.time()-started,3),
                    'model_key': model_key, 'model_id': spec['model_id'],
                })
                write_jsonl(model_dir/f'{task}.jsonl', preds)
        RUN_STATUS.append({'model_key':model_key,'model_id':spec['model_id'],'status':'completed'})
    except Exception as exc:
        traceback.print_exc()
        RUN_STATUS.append({'model_key':model_key,'model_id':spec['model_id'],'status':'load_or_run_failed','error':str(exc)})
    finally:
        try: del pipe
        except Exception: pass
        gc.collect()
        if torch.cuda.is_available(): torch.cuda.empty_cache()

json.dump(RUN_STATUS, open(RUN_ROOT/'run_status.json','w'), ensure_ascii=False, indent=2)
print(RUN_STATUS)


In [ ]:
# 10) 기존 evaluator를 모델별로 실행
for status in RUN_STATUS:
    if status['status'] != 'completed': continue
    key=status['model_key']; d=RUN_ROOT/key; metrics=d/'metrics'; metrics.mkdir(exist_ok=True)
    commands=[
        [sys.executable,'scripts/evaluate.py','--task','anomaly','--qa','samples/anomaly_zero_shot_20.jsonl','--predictions',str(d/'zero_shot.jsonl'),'--regime','zero_shot','--out-dir',str(metrics)],
        [sys.executable,'scripts/evaluate.py','--task','anomaly','--qa','samples/anomaly_one_shot_20.jsonl','--predictions',str(d/'one_shot.jsonl'),'--regime','one_shot','--out-dir',str(metrics)],
        [sys.executable,'scripts/evaluate.py','--task','explainability','--qa','samples/defect_20.jsonl','--predictions',str(d/'structured.jsonl'),'--out-dir',str(metrics)],
    ]
    for cmd in commands: subprocess.run(cmd, check=True)
print('evaluation done')


In [ ]:
# 11) 모델 비교 leaderboard 생성
import pandas as pd
rows=[]
for status in RUN_STATUS:
    if status['status'] != 'completed': continue
    key=status['model_key']; spec=MODEL_REGISTRY[key]; m=RUN_ROOT/key/'metrics'
    anomaly=pd.read_csv(m/'anomaly_overall.csv')
    explain=pd.read_csv(m/'explainability_overall.csv').iloc[0]
    z=anomaly[anomaly['regime']=='zero_shot'].iloc[0]
    o=anomaly[anomaly['regime']=='one_shot'].iloc[0]
    rows.append({
        'model_key':key, 'family':spec['family'], 'model_id':spec['model_id'],
        'zero_bacc':z['balanced_accuracy'], 'zero_f1':z['f1'], 'zero_invalid':z['invalid_output_rate'],
        'one_bacc':o['balanced_accuracy'], 'one_f1':o['f1'], 'one_invalid':o['invalid_output_rate'],
        'defect_bacc':explain['defect_balanced_accuracy'], 'defect_macro_f1':explain['defect_macro_f1'],
        'joint_accuracy':explain['joint_accuracy'], 'false_defect_rate':explain['false_defect_rate'],
        'consistency_error':explain['consistency_error'], 'structured_invalid':explain['invalid_output_rate'],
    })
leaderboard=pd.DataFrame(rows)
leaderboard.to_csv(RUN_ROOT/'leaderboard.csv',index=False)
display(leaderboard.sort_values(['joint_accuracy','one_bacc'],ascending=False))


In [ ]:
# 12) 사람이 보기 쉬운 prediction review CSV
review=[]
for status in RUN_STATUS:
    if status['status']!='completed': continue
    key=status['model_key']; d=RUN_ROOT/key
    for task, qa_rows in QA_SETS.items():
        qa={r['id']:r for r in qa_rows}
        for p in read_jsonl(d/f'{task}.jsonl'):
            gt=qa[p['id']]['answer']
            review.append({
                'model_key':key,'task':task,'id':p['id'],
                'asset_category':qa[p['id']]['asset_category'],
                'gt':json.dumps(gt,ensure_ascii=False),
                'output':p['output'],'error':p.get('error'),'latency_s':p.get('latency_s')
            })
pd.DataFrame(review).to_csv(RUN_ROOT/'prediction_review.csv',index=False)
print('saved:',RUN_ROOT/'prediction_review.csv')


In [ ]:
# 13) 실행 메타데이터 + 결과 ZIP 묶기
metadata={
    'branch':BRANCH,
    'gpu':torch.cuda.get_device_name(0),
    'vram_gb':round(VRAM_GB,2),
    'models_requested':MODELS_TO_RUN,
    'model_registry':MODEL_REGISTRY,
    'qa_counts':{k:len(v) for k,v in QA_SETS.items()},
    'decoding':{'do_sample':False,'max_new_tokens':80},
}
(RUN_ROOT/'run_metadata.json').write_text(json.dumps(metadata,ensure_ascii=False,indent=2),encoding='utf-8')
archive=shutil.make_archive(str(RUN_ROOT),'zip',root_dir=RUN_ROOT)
print('RESULT ZIP:',archive)
print('이 ZIP 하나를 ChatGPT에 올리면 모델별 성능/오답 비교 분석 가능')
